# Tune the cuts and match filters on known dwarf galaxies and globular clusters

A known object at the distance modulus of interest is a real old, metal-poor population: the signal of a
selection (cuts, colour or magnitude ranges, a match filter in some bands) is its excess of stars on the object
over the field around it. Module `analysis.gc_dwarf_optimization`, plots in `plotting.plot_gc_dwarf_optimization`.

Every selection is a boolean array over the rows of the catalog: combine them with `&` and `|`, with any cut on
the catalog columns. The catalog magnitudes are already dereddened.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from streameucliddes.analysis.gc_dwarf_optimization import (
    compare_selections,
    get_object,
    object_field_radius_deg,
    object_isochrone,
    object_radec_box,
    object_regions,
    objects_near_distance_modulus,
    selection_snr,
    stars_in_match_filter,
    stars_passing_cuts,
)
from streameucliddes.catalogs.euclid_catalog import default_star_cuts, get_euclid_stars
from streameucliddes.plotting.plot_gc_dwarf_optimization import (
    plot_object_cmd,
    plot_object_hess,
    plot_object_map,
    plot_selection_comparison,
)

## 1. Known objects near a distance modulus

In [ ]:
DM_OF_INTEREST = 17.0
MAX_DM_OFFSET = 1.0
CATALOG = "euclid_dr1"  # Euclid DR1 stars, Euclid bands only
STAR_SELECTION = "morphology_cuts"  # or "xgboost_classifier" (see default_star_cuts)

# in_catalog: sources of the catalog at the centre of the object (HATS point map, no star read)
objects = objects_near_distance_modulus(DM_OF_INTEREST, MAX_DM_OFFSET, object_types=("dwarfs", "gc"), catalog=CATALOG)
objects[objects["in_catalog"]][["type", "key", "name", "radius_arcmin", "ellipticity", "distance_modulus", "dm_offset",
                                "feh", "n_sources_at_centre"]]

## 2. One object and its field

The on region is an ellipse (or an elliptical annulus) between `ON_RADII` half-light radii, the off region an
elliptical annulus between `OFF_RADII`. For a globular cluster, start the on annulus beyond its core, where the
catalog misses stars (crowding), and the off annulus beyond its stars: e.g. NGC 1851, `ON_RADII = (4, 15)`,
`OFF_RADII = (30, 40)`.

In [ ]:
OBJECT_NAME = "Reticulum II"
ON_RADII = (0.0, 2.0)  # half-light radii
OFF_RADII = (5.0, 10.0)

obj = get_object(OBJECT_NAME)
isochrone = object_isochrone(obj)  # LVDB [Fe/H] as Z, old age
distance_modulus = obj["distance_modulus"]
print(obj[["name", "radius_arcmin", "ellipticity", "position_angle", "distance_modulus", "feh"]].to_dict())
print("isochrone:", isochrone)

ra_range, dec_range = object_radec_box(obj, object_field_radius_deg(obj, OFF_RADII))
catalog = get_euclid_stars(CATALOG, cuts=default_star_cuts(CATALOG, STAR_SELECTION), ra_range=ra_range,
                           dec_range=dec_range)
print([column for column in catalog.columns if column.endswith("_obs")])

In [ ]:
regions = object_regions(catalog, obj, on_radii=ON_RADII, off_radii=OFF_RADII)
print(f"on: {regions['on'].sum()} stars, {regions['on_area_deg2']:.4f} deg2 ({regions['on_area_fraction']:.0%} kept)")
print(f"off: {regions['off'].sum()} stars, {regions['off_area_deg2']:.4f} deg2 ({regions['off_area_fraction']:.0%} kept)")
print(f"{len(regions['masked_objects'])} other objects masked in the field")
fig, ax = plot_object_map(regions, smoothing_bins=1)  # all the stars

## 3. Match filters: several band combinations

In [ ]:
MATCH_FILTERS = {
    "VIS-H": {"name": "vis_h", "reference_band": ("euclid_dr1", "VIS"), "color_bands": [("euclid_dr1", "H")]},
    "VIS-J": {"name": "vis_j", "reference_band": ("euclid_dr1", "VIS"), "color_bands": [("euclid_dr1", "J")]},
    "VIS-Y": {"name": "vis_y", "reference_band": ("euclid_dr1", "VIS"), "color_bands": [("euclid_dr1", "Y")]},
    "VIS-Y,J,H": {"name": "vis_yjh", "reference_band": ("euclid_dr1", "VIS"),
                  "color_bands": [("euclid_dr1", "Y"), ("euclid_dr1", "J"), ("euclid_dr1", "H")]},
}
selections = {name: stars_in_match_filter(catalog, distance_modulus, isochrone, match_filter)
              for name, match_filter in MATCH_FILTERS.items()}
table = compare_selections(regions, selections)
display(table.round(2))
fig, ax = plot_selection_comparison(table)

In [ ]:
BEST = "VIS-H"
fig, ax = plot_object_map(regions, selections[BEST], smoothing_bins=1)

## 4. CMDs and Hess diagrams

One panel (CMD) or one row (Hess: on, off, on - off) per colour of the match filter. `stars`: the stars drawn,
here those passing the error cuts of the match filter.

In [ ]:
match_filter = MATCH_FILTERS["VIS-Y,J,H"]
shown = stars_passing_cuts(catalog, match_filter)
fig, axes = plot_object_cmd(catalog, regions, distance_modulus, isochrone, match_filter, stars=shown)

In [ ]:
fig, axes = plot_object_hess(catalog, regions, distance_modulus, isochrone, match_filter, stars=shown)

## 5. Your own cuts at the catalog level

Any expression on the catalog columns, as a boolean array, combined with a match filter (`mask=`) or with
another selection (`&`).

In [ ]:
vis, h = catalog["euclid_dr1_VIS_obs"].to_numpy(), catalog["euclid_dr1_H_obs"].to_numpy()
my_selections = {
    "VIS-H": selections["VIS-H"],
    "VIS-H, VIS < 24": selections["VIS-H"] & (vis < 24.0),
    "VIS-H, VIS < 25": selections["VIS-H"] & (vis < 25.0),
    "VIS-H, errors < 0.1": stars_in_match_filter(catalog, distance_modulus, isochrone, MATCH_FILTERS["VIS-H"],
                                                 cuts={"euclid_dr1_VIS_err": (None, 0.1), "euclid_dr1_H_err": (None, 0.1)}),
    "colour box only": (vis - h > -0.5) & (vis - h < 0.3) & (vis < 25.0),
}
my_table = compare_selections(regions, my_selections)
display(my_table.round(2))
fig, ax = plot_selection_comparison(my_table)

In [ ]:
# CMDs and Hess diagrams of any band pairs (colour = first - second), without a match filter
pairs = [("euclid_dr1_Y_obs", "euclid_dr1_H_obs"), ("euclid_dr1_J_obs", "euclid_dr1_H_obs")]
fig, axes = plot_object_cmd(catalog, regions, mag_cols=pairs)
fig, axes = plot_object_hess(catalog, regions, mag_cols=pairs, color_bins=np.arange(-0.5, 1.0, 0.04),
                             mag_bins=np.arange(18.0, 26.0, 1 / 6))